# Bilstm (w/o features)

In [ ]:
# =========================================================
# 3-CLASS VEDIC ACCENT PREDICTION (WEIGHTED, TOKEN LEVEL)
# USING YOUR VEDAWEB DATASET
# =========================================================

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
import unicodedata

# -------------------------
# PARAMETERS
# -------------------------
MAX_LEN = 32
CHAR_EMB_DIM = 64
LSTM_UNITS = 128
DROPOUT_RATE = 0.3
BATCH_SIZE = 64
EPOCHS = 10
TEST_SIZE = 0.2

# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_excel("finalRoorkeeCorpus.xlsx")

words_accented = df["docu::form"].astype(str).tolist()
words_clean = df["docu::form_clean"].astype(str).tolist()

# -------------------------
# EXTRACT LABELS (0=no accent, 1=acute, 2=grave)
# -------------------------
def extract_labels(accented, clean):
    labels = []
    j = 0
    for c in accented:
        if unicodedata.combining(c):
            if c == "\u0301":      # acute
                labels[-1] = 1
            elif c == "\u0300":    # grave
                labels[-1] = 2
        else:
            labels.append(0)
            j += 1
    return labels

all_labels = []
for a, c in zip(words_accented, words_clean):
    lab = extract_labels(a, c)
    if len(lab) == len(c):
        all_labels.append(lab)
    else:
        all_labels.append([0]*len(c))  # fallback safety

# -------------------------
# BUILD CHAR VOCAB FROM CLEAN WORDS
# -------------------------
chars = set("".join(words_clean))
char2idx = {c: i+1 for i, c in enumerate(sorted(chars))}
char2idx["PAD"] = 0

VOCAB_SIZE = len(char2idx)

print("Vocab size:", VOCAB_SIZE)

# -------------------------
# ENCODE INPUTS
# -------------------------
X = [[char2idx[c] for c in w] for w in words_clean]
X = pad_sequences(X, maxlen=MAX_LEN, padding="post")

y = pad_sequences(all_labels, maxlen=MAX_LEN, padding="post")
y = np.expand_dims(y, -1)

# -------------------------
# SPLIT
# -------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42
)

# -------------------------
# COMPUTE TOKEN-LEVEL CLASS WEIGHTS
# -------------------------
mask = X_train != 0
y_train_flat = y_train.squeeze()[mask]

classes = np.unique(y_train_flat)

class_weights = compute_class_weight(
    class_weight="balanced", #####################################################
    classes=classes,
    y=y_train_flat
)

class_weights = dict(zip(classes, class_weights))
print("Class weights:", class_weights)

weights_tensor = tf.constant(
    [class_weights.get(i, 1.0) for i in range(3)],
    dtype=tf.float32
)

# -------------------------
# CUSTOM WEIGHTED LOSS
# -------------------------
def weighted_loss(y_true, y_pred):
    y_true = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)

    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)

    weights = tf.gather(weights_tensor, y_true)
    loss = loss * weights

    return tf.reduce_mean(loss)

# -------------------------
# MODEL
# -------------------------
inputs = Input(shape=(MAX_LEN,))
x = Embedding(VOCAB_SIZE, CHAR_EMB_DIM, mask_zero=True)(inputs)
x = Bidirectional(LSTM(LSTM_UNITS, return_sequences=True))(x)
x = Dropout(DROPOUT_RATE)(x)
outputs = Dense(3, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss=weighted_loss,
    metrics=["accuracy"]
)

model.summary()

# -------------------------
# TRAIN
# -------------------------
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

# -------------------------
# EVALUATION
# -------------------------
y_pred = model.predict(X_val)
y_pred = np.argmax(y_pred, axis=-1)

y_true = y_val.squeeze()

mask = X_val != 0
y_true_flat = y_true[mask]
y_pred_flat = y_pred[mask]

print("\nClassification Report:\n")
print(classification_report(y_true_flat, y_pred_flat))

print("Macro F1:", f1_score(y_true_flat, y_pred_flat, average="macro"))


# =========================================================
# SAVE XLSX WITH PREDICTED WORD COLUMN
# =========================================================

# --- Split again but KEEP indices ---
X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X, y, np.arange(len(X)),
    test_size=TEST_SIZE,
    random_state=42
)

# --- Predict on FULL dataset ---
full_preds = model.predict(X)
full_preds = np.argmax(full_preds, axis=-1)

# --- Function to reconstruct accented word ---
def reconstruct_word(clean_word, pred_labels):
    result = ""
    for ch, lab in zip(clean_word, pred_labels):
        result += ch
        if lab == 1:
            result += "\u0301"  # acute accent
        elif lab == 2:
            result += "\u0300"  # grave accent
    return result

# --- Build predicted words ---
predicted_words = []
for word, pred in zip(words_clean, full_preds):
    predicted_words.append(reconstruct_word(word, pred))

# --- Create new columns ---
pred_column = ["NA"] * len(df)
pred_word_column = ["NA"] * len(df)

# Fill ONLY validation rows
for i in idx_val:
    pred_column[i] = "VAL"
    pred_word_column[i] = predicted_words[i]

# --- Insert at beginning of dataframe ---
df.insert(0, "pred", pred_column)
df.insert(1, "predicted_word", pred_word_column)

# --- Save file ---
df.to_excel("pred_bilstm.xlsx", index=False)

print("------------------------------------------------")

Vocab size: 39
Class weights: {np.int32(0): np.float64(0.3733514480156756), np.int32(1): np.float64(3.164988966476225), np.int32(2): np.float64(178.51832907075874)}


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 32, 64)    │      2,496 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 32)        │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 32, 256)   │    197,632 │ embedding[0][0],  │
│ (Bidirectional)     │                   │            │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32, 256)   │          0 │ bidirectional[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32, 3)     │        771 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 200,899 (784.76 KB)

 Trainable params: 200,899 (784.76 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
428/428 ━━━━━━━━━━━━━━━━━━━━ 68s 145ms/step - accuracy: 0.9283 - loss: 0.4697 - val_accuracy: 0.9271 - val_loss: 0.2741
Epoch 2/10
428/428 ━━━━━━━━━━━━━━━━━━━━ 61s 142ms/step - accuracy: 0.9291 - loss: 0.2575 - val_accuracy: 0.9270 - val_loss: 0.1970
Epoch 3/10
428/428 ━━━━━━━━━━━━━━━━━━━━ 62s 145ms/step - accuracy: 0.9295 - loss: 0.1864 - val_accuracy: 0.9278 - val_loss: 0.1591
Epoch 4/10
428/428 ━━━━━━━━━━━━━━━━━━━━ 62s 145ms/step - accuracy: 0.9337 - loss: 0.1460 - val_accuracy: 0.9372 - val_loss: 0.1252
Epoch 5/10
428/428 ━━━━━━━━━━━━━━━━━━━━ 63s 147ms/step - accuracy: 0.9371 - loss: 0.1208 - val_accuracy: 0.9392 - val_loss: 0.1066
Epoch 6/10
428/428 ━━━━━━━━━━━━━━━━━━━━ 62s 145ms/step - accuracy: 0.9374 - loss: 0.1063 - val_accuracy: 0.9403 - val_loss: 0.0958
Epoch 7/10
428/428 ━━━━━━━━━━━━━━━━━━━━ 64s 149ms/step - accuracy: 0.9395 - loss: 0.0966 - val_accuracy: 0.9385 - val_loss: 0.0874
Epoch 8/10
428/428 ━━━━━━━━━━━━━━━━━━━━ 64s 150ms/step - accuracy: 0.9401 - loss: 0

In [ ]:
'''
# =========================================================
# ADD PREDICTED ACCENTED WORDS TO EXCEL (CHAR-ONLY MODEL)
# =========================================================

# Predict on FULL dataset (not just validation)
all_preds = model.predict(X)
all_preds = np.argmax(all_preds, axis=-1)

# Helper: rebuild accented word
def rebuild_accented(clean_word, pred_labels):
    result = ""
    for ch, label in zip(clean_word, pred_labels):
        result += ch
        if label == 1:
            result += "\u0301"  # acute
        elif label == 2:
            result += "\u0300"  # grave
    return result

predicted_words = []

for word, preds in zip(words_clean, all_preds):
    preds_trimmed = preds[:len(word)]  # remove padding part
    predicted_words.append(rebuild_accented(word, preds_trimmed))

# Insert as FIRST column
df.insert(0, "PREDICTED_ACCENT", predicted_words)

# Save new file
df.to_excel("Vedaweb_with_predictions_char_only.xlsx", index=False)

print("File saved as: Vedaweb_with_predictions_char_only.xlsx")
'''

1069/1069 ━━━━━━━━━━━━━━━━━━━━ 31s 29ms/step


ValueError: cannot insert PREDICTED_ACCENT, already exists

In [ ]:
'''
Classification Report:   bilstm (no feats) 80:20 correct

              precision    recall  f1-score   support

           0       0.99      0.74      0.85     46768
           1       0.30      0.92      0.46      5505
           2       0.12      0.97      0.21       108

    accuracy                           0.76     52381
   macro avg       0.47      0.88      0.51     52381
weighted avg       0.92      0.76      0.81     52381

Macro F1: 0.5063961897578356


Classification Report: correct corpus - paper grade

              precision    recall  f1-score   support

           0       0.99      0.76      0.86     46684
           1       0.32      0.92      0.48      5589
           2       0.15      0.99      0.26       108

    accuracy                           0.78     52381
   macro avg       0.49      0.89      0.53     52381
weighted avg       0.92      0.78      0.82     52381

Macro F1: 0.5313883707471604
'''

# transformer (w/o features + embedded input (not one hot))

In [1]:
# =========================================================
# 3-CLASS VEDIC ACCENT PREDICTION (WEIGHTED, TOKEN LEVEL)
# TRANSFORMER (CHAR-ONLY)
# =========================================================

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Dense, Dropout,
    LayerNormalization, MultiHeadAttention
)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
import unicodedata

# -------------------------
# PARAMETERS
# -------------------------
MAX_LEN = 32
CHAR_EMB_DIM = 64
NUM_HEADS = 4
FF_DIM = 128
NUM_TRANSFORMER_BLOCKS = 2
DROPOUT_RATE = 0.3
BATCH_SIZE = 64
EPOCHS = 10
TEST_SIZE  = 0.2

# -------------------------
# LOAD DATA
# -------------------------
df = pd.read_excel("finalRoorkeeCorpus.xlsx")

words_accented = df["docu::form"].astype(str).tolist()
words_clean = df["docu::form_clean"].astype(str).tolist()

# -------------------------
# EXTRACT LABELS
# -------------------------
def extract_labels(accented, clean):
    labels = []
    for c in accented:
        if unicodedata.combining(c):
            if c == "\u0301":
                labels[-1] = 1
            elif c == "\u0300":
                labels[-1] = 2
        else:
            labels.append(0)
    return labels

all_labels = []
for a, c in zip(words_accented, words_clean):
    lab = extract_labels(a, c)
    if len(lab) == len(c):
        all_labels.append(lab)
    else:
        all_labels.append([0]*len(c))

# -------------------------
# BUILD CHAR VOCAB
# -------------------------
chars = set("".join(words_clean))
char2idx = {c: i+1 for i, c in enumerate(sorted(chars))}
char2idx["PAD"] = 0
VOCAB_SIZE = len(char2idx)

print("Vocab size:", VOCAB_SIZE)

# -------------------------
# ENCODE INPUTS
# -------------------------
X = [[char2idx[c] for c in w] for w in words_clean]
X = pad_sequences(X, maxlen=MAX_LEN, padding="post")

y = pad_sequences(all_labels, maxlen=MAX_LEN, padding="post")
y = np.expand_dims(y, -1)

# -------------------------
# SPLIT
# -------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=42
)

# -------------------------
# TOKEN-LEVEL CLASS WEIGHTS
# -------------------------
mask = X_train != 0
y_train_flat = y_train.squeeze()[mask]

classes = np.unique(y_train_flat)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_flat
)

class_weights = dict(zip(classes, class_weights))
print("Class weights:", class_weights)

weights_tensor = tf.constant(
    [class_weights.get(i, 1.0) for i in range(3)],
    dtype=tf.float32
)

def weighted_loss(y_true, y_pred):
    y_true = tf.cast(tf.squeeze(y_true, axis=-1), tf.int32)
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    weights = tf.gather(weights_tensor, y_true)
    return tf.reduce_mean(loss * weights)

# -------------------------
# POSITIONAL EMBEDDING
# -------------------------
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = Embedding(vocab_size, embed_dim)
        self.pos_emb = Embedding(max_len, embed_dim)
        self.max_len = max_len

    def call(self, x):
        positions = tf.range(start=0, limit=self.max_len, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

# -------------------------
# TRANSFORMER BLOCK
# -------------------------
def transformer_block(x):
    attn_output = MultiHeadAttention(
        num_heads=NUM_HEADS,
        key_dim=CHAR_EMB_DIM
    )(x, x)

    attn_output = Dropout(DROPOUT_RATE)(attn_output)
    x = LayerNormalization(epsilon=1e-6)(x + attn_output)

    ffn = Dense(FF_DIM, activation="relu")(x)
    ffn = Dense(CHAR_EMB_DIM)(ffn)
    ffn = Dropout(DROPOUT_RATE)(ffn)

    return LayerNormalization(epsilon=1e-6)(x + ffn)

# -------------------------
# MODEL
# -------------------------
inputs = Input(shape=(MAX_LEN,))

x = PositionalEmbedding(MAX_LEN, VOCAB_SIZE, CHAR_EMB_DIM)(inputs)

for _ in range(NUM_TRANSFORMER_BLOCKS):
    x = transformer_block(x)

x = Dropout(DROPOUT_RATE)(x)

outputs = Dense(3, activation="softmax")(x)

model = Model(inputs, outputs)

model.compile(
    optimizer="adam",
    loss=weighted_loss,
    metrics=["accuracy"]
)

model.summary()

# -------------------------
# TRAIN
# -------------------------
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE
)

# -------------------------
# EVALUATION
# -------------------------
y_pred = model.predict(X_val)
y_pred = np.argmax(y_pred, axis=-1)

y_true = y_val.squeeze()

mask = X_val != 0
y_true_flat = y_true[mask]
y_pred_flat = y_pred[mask]

print("\nClassification Report:\n")
print(classification_report(y_true_flat, y_pred_flat))

print("Macro F1:", f1_score(y_true_flat, y_pred_flat, average="macro"))



# =========================================================
# SAVE XLSX WITH PREDICTED WORD COLUMN
# =========================================================

# --- Split again but KEEP indices ---
X_train, X_val, y_train, y_val, idx_train, idx_val = train_test_split(
    X, y, np.arange(len(X)),
    test_size=TEST_SIZE,
    random_state=42
)

# --- Predict on FULL dataset ---
full_preds = model.predict(X)
full_preds = np.argmax(full_preds, axis=-1)

# --- Function to reconstruct accented word ---
def reconstruct_word(clean_word, pred_labels):
    result = ""
    for ch, lab in zip(clean_word, pred_labels):
        result += ch
        if lab == 1:
            result += "\u0301"  # acute
        elif lab == 2:
            result += "\u0300"  # grave
    return result

# --- Build predicted words ---
predicted_words = []
for word, pred in zip(words_clean, full_preds):
    predicted_words.append(reconstruct_word(word, pred))

# --- Create columns ---
pred_column = ["NA"] * len(df)
pred_word_column = ["NA"] * len(df)

# Fill ONLY validation rows
for i in idx_val:
    pred_column[i] = "VAL"
    pred_word_column[i] = predicted_words[i]

# --- Insert at beginning ---
df.insert(0, "pred", pred_column)
df.insert(1, "predicted_word", pred_word_column)

# --- Save ---
df.to_excel("xmer_pred_nofeats.xlsx", index=False)

print("Saved file: .xlsx")

Vocab size: 39
Class weights: {np.int32(0): np.float64(0.3733514480156756), np.int32(1): np.float64(3.164988966476225), np.int32(2): np.float64(178.51832907075874)}


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ positional_embeddi… │ (None, 32, 64)    │      4,544 │ input_layer[0][0] │
│ (PositionalEmbeddi… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 32, 64)    │     66,368 │ positional_embed… │
│ (MultiHeadAttentio… │                   │            │ positional_embed… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 32, 64)    │          0 │ positional_embed… │
│                     │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 32, 64)    │        128 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32, 64)    │      8,256 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 32, 64)    │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 32, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 32, 64)    │        128 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 32, 64)    │     66,368 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 32, 64)    │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 32, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 32, 64)    │        128 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 32, 64)    │      8,256 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 32, 64)    │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 32, 64)    │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 171,139 (668.51 KB)

 Trainable params: 171,139 (668.51 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
428/428 ━━━━━━━━━━━━━━━━━━━━ 93s 200ms/step - accuracy: 0.9177 - loss: 0.2373 - val_accuracy: 0.9219 - val_loss: 0.1700
Epoch 2/10
428/428 ━━━━━━━━━━━━━━━━━━━━ 77s 181ms/step - accuracy: 0.9210 - loss: 0.1627 - val_accuracy: 0.9225 - val_loss: 0.1183
Epoch 3/10
 17/428 ━━━━━━━━━━━━━━━━━━━━ 1:00 147ms/step - accuracy: 0.9225 - loss: 0.1118

KeyboardInterrupt: 

In [ ]:
'''
# =========================================================
# ADD PREDICTED ACCENTED WORDS TO EXCEL (TRANSFORMER MODEL)
# =========================================================

from google.colab import files

# Predict on FULL dataset
all_preds = model.predict(X)
all_preds = np.argmax(all_preds, axis=-1)

# Helper: rebuild accented word
def rebuild_accented(clean_word, pred_labels):
    result = ""
    for ch, label in zip(clean_word, pred_labels):
        result += ch
        if label == 1:
            result += "\u0301"  # acute
        elif label == 2:
            result += "\u0300"  # grave
    return result

predicted_words = []

for word, preds in zip(words_clean, all_preds):
    preds_trimmed = preds[:len(word)]  # remove padding
    predicted_words.append(rebuild_accented(word, preds_trimmed))

# Remove old prediction column if already exists
if "PREDICTED_ACCENT" in df.columns:
    df.drop(columns=["PREDICTED_ACCENT"], inplace=True)

# Insert as FIRST column
df.insert(0, "PREDICTED_ACCENT", predicted_words)

# Save file
file_name = "Vedaweb_with_predictions_transformer.xlsx"
df.to_excel(file_name, index=False)

print("File saved as:", file_name)

# Auto-download
files.download(file_name)
'''

1069/1069 ━━━━━━━━━━━━━━━━━━━━ 28s 27ms/step
File saved as: Vedaweb_with_predictions_transformer.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
'''
Classification Report:

              precision    recall  f1-score   support

           0       1.00      0.66      0.80     46684
           1       0.26      0.84      0.40      5589
           2       0.03      0.95      0.05       108

    accuracy                           0.68     52381
   macro avg       0.43      0.82      0.42     52381
weighted avg       0.92      0.68      0.75     52381

Macro F1: 0.4172626251448674


'''